# SamplerV2 Sample Programs
### Modified for Quantum Rings toolkit for Qiskit 2.x
#### See here: https://qiskit.qotlabs.org/docs/guides/primitives-examples

### Install Required Packages

In [ ]:
%%capture
%pip install quantumrings-toolkit-qiskit
%pip install qiskit

In [1]:
#
# Setup your account
# You can also save your account locally using the class method QrRuntimeService.save_account(...) and
# invoke the QrRuntimeService class constructor without any arguments.
#
import os
my_token = os.environ["QR_TOKEN"]
my_name = os.environ["QR_ACCOUNT"]

#
# Set the backend of your choice, depending upon the task and your hardware configuration.
# See SDK documentation for additional help.
#

my_backend = "scarlet_quantum_rings"

## Run a single experiment

In [2]:
import numpy as np
from qiskit.circuit.library import IQP
from qiskit.quantum_info import random_hermitian
#from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
 
#service = QiskitRuntimeService()
 
#backend = service.least_busy(operational=True, simulator=False, min_num_qubits=127)

from quantumrings.toolkit.qiskit import QrRuntimeService
from quantumrings.toolkit.qiskit import QrSamplerV2 as Sampler

service = QrRuntimeService( token = my_token, name = my_name)
avaiable_backends = service.backends()
print(avaiable_backends)
#backend = service.least_busy(operational=True, simulator=False)

n_qubits = 12#7

backend = service.backend(name = my_backend, precision = "single", gpu = 0, num_qubits = n_qubits)
 
mat = np.real(random_hermitian(n_qubits, seed=1234))
circuit = IQP(mat)
circuit.measure_all()
 
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(circuit)
 
sampler = Sampler(backend = backend)
job = sampler.run([isa_circuit])
result = job.result()

pub_result = result[0]
 
print(f" > First ten results: {pub_result.data.meas.get_bitstrings()[:10]}")

['scarlet_quantum_rings']


C:\Users\vkasi\AppData\Local\Temp\ipykernel_47772\881852986.py:24: DeprecationWarning: The class ``qiskit.circuit.library.iqp.IQP`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the qiskit.circuit.library.iqp function instead.
  circuit = IQP(mat)


 > First ten results: ['000000000000', '000000000001', '010000001001', '000000000001', '000000000001', '000001001001', '000000000001', '000010101000', '110000000001', '010010100001']


## Run multiple experiments in a single job

In [3]:
print(pub_result.data.meas)

BitArray(<shape=(1,), num_shots=1024, num_bits=12>)


In [4]:
import numpy as np
from qiskit.circuit.library import iqp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import random_hermitian
#from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

from quantumrings.toolkit.qiskit import QrRuntimeService
from quantumrings.toolkit.qiskit import QrSamplerV2 as Sampler

n_qubits = 12 #7
 
#service = QiskitRuntimeService()
#backend = service.least_busy(
#    operational=True, simulator=False, min_num_qubits=n_qubits
#)

service = QrRuntimeService( token = my_token, name = my_name)
avaiable_backends = service.backends()
print(avaiable_backends)

backend = service.backend(name = my_backend, precision = "single", gpu = 0, num_qubits = n_qubits)
 
rng = np.random.default_rng()
mats = [np.real(random_hermitian(n_qubits, seed=rng)) for _ in range(3)]
circuits = [iqp(mat) for mat in mats]
for circuit in circuits:
    circuit.measure_all()
 
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuits = pm.run(circuits)
 
sampler = Sampler(backend = backend)
job = sampler.run(isa_circuits)
result = job.result()
 
for idx, pub_result in enumerate(result):
    print(
        f" > First ten results for pub {idx}: {pub_result.data.meas.get_bitstrings()[:10]}"
    )

['scarlet_quantum_rings']
 > First ten results for pub 0: ['000000000000', '000000000010', '111000000010', '101000100000', '100100100000', '110000000000', '000000000000', '010000000010', '100000000000', '000000000000']
 > First ten results for pub 1: ['100100000010', '011000010010', '001000000000', '001000000100', '001000000000', '101000000000', '000000000010', '010000000000', '001000010000', '000000000110']
 > First ten results for pub 2: ['000000000010', '000000000000', '010000000010', '000000000010', '000000000010', '000000000010', '000000000000', '000000100000', '100000000010', '000000100000']


## Run parameterized circuits

In [5]:
import numpy as np
from qiskit.circuit.library import real_amplitudes
from qiskit.transpiler import generate_preset_pass_manager
#from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler


from quantumrings.toolkit.qiskit import QrRuntimeService
from quantumrings.toolkit.qiskit import QrSamplerV2 as Sampler

n_qubits = 12#7
 
#service = QiskitRuntimeService()
#backend = service.least_busy(
#    operational=True, simulator=False, min_num_qubits=n_qubits
#)

service = QrRuntimeService( token = my_token, name = my_name)
backend = service.backend(name = my_backend, precision = "single", gpu = 0, num_qubits = n_qubits)

# Step 1: Map classical inputs to a quantum problem
circuit = real_amplitudes(num_qubits=n_qubits, reps=2)
circuit.measure_all()

# Define three sets of parameters for the circuit
rng = np.random.default_rng(1234)
parameter_values = [
    rng.uniform(-np.pi, np.pi, size=circuit.num_parameters) for _ in range(3)
]
 
# Step 2: Optimize problem for quantum execution.
 
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(circuit)
 
# Step 3: Execute using Qiskit primitives.
sampler = Sampler(backend = backend)

job = sampler.run([(isa_circuit, parameter_values)])
result = job.result()
# Get results for the first (and only) PUB
pub_result = result[0]
# Get counts from the classical register "meas".
print(
    f" >> First ten results for the meas output register: {pub_result.data.meas.get_bitstrings()[:10]}"
)

 >> First ten results for the meas output register: ['101000100111', '000100000101', '101000000111', '110100100011', '110001101101', '010100101111', '000000011111', '100101000011', '001100100101', '110011000011']


## Use sessions and advanced options

In [6]:
import numpy as np
from qiskit.circuit.library import iqp
from qiskit.quantum_info import random_hermitian
from qiskit.transpiler import generate_preset_pass_manager
#from qiskit_ibm_runtime import Session, SamplerV2 as Sampler
#from qiskit_ibm_runtime import QiskitRuntimeService

from quantumrings.toolkit.qiskit import QrRuntimeService
from quantumrings.toolkit.qiskit import QrSamplerV2 as Sampler
from quantumrings.toolkit.qiskit import QrSession as Session

n_qubits = 12 #7
 
#service = QiskitRuntimeService()
#backend = service.least_busy(
#    operational=True, simulator=False, min_num_qubits=n_qubits
#)

service = QrRuntimeService( token = my_token, name = my_name)
backend = service.backend(name = my_backend, precision = "single", gpu = 0, num_qubits = n_qubits)


rng = np.random.default_rng(1234)
mat = np.real(random_hermitian(n_qubits, seed=rng))
circuit = iqp(mat)
circuit.measure_all()
mat = np.real(random_hermitian(n_qubits, seed=rng))
another_circuit = iqp(mat)
another_circuit.measure_all()
 
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(circuit)
another_isa_circuit = pm.run(another_circuit)
 
with Session(backend=backend) as session:
    sampler = Sampler(mode=session)
    job = sampler.run([isa_circuit])
    another_job = sampler.run([another_isa_circuit])
    result = job.result()
    another_result = another_job.result()
 
# first job
 
print(
    f" > The first ten measurement results of job 1: {result[0].data.meas.get_bitstrings()[:10]}"
)

# second job
print(
    " > The first ten measurement results of job 2:",
    another_result[0].data.meas.get_bitstrings()[:10],
)

 > The first ten measurement results of job 1: ['000110101001', '000000000001', '000100100000', '000000100000', '000101000001', '000000000000', '000000000001', '000000000001', '010000100000', '100010001001']
 > The first ten measurement results of job 2: ['011011010110', '010001010011', '000110000001', '000010010010', '000011000000', '011000000110', '001000000100', '000110010001', '000111000101', '010100001010']


## Multiple Classical Register Usage

In [7]:
from qiskit.circuit import (
    Parameter, QuantumCircuit, ClassicalRegister, QuantumRegister
)

#from qiskit.primitives import StatevectorSampler
#from quantumrings.toolkit.qiskit import QrStatevectorSampler
from quantumrings.toolkit.qiskit import QrRuntimeService
from quantumrings.toolkit.qiskit import QrSamplerV2

import matplotlib.pyplot as plt
import numpy as np

# Acquire Quantum Rings backend
qr_services = QrRuntimeService(name = my_name, token = my_token)
qr_backend = qr_services.backend(name = my_backend, precision = "single")

# Define our circuit registers, including classical registers
# called 'alpha' and 'beta'.
qreg = QuantumRegister(3)
alpha = ClassicalRegister(2, "alpha")
beta = ClassicalRegister(1, "beta")
 
# Define a quantum circuit with two parameters.
circuit = QuantumCircuit(qreg, alpha, beta)
circuit.h(0)
circuit.cx(0, 1)
circuit.cx(1, 2)
circuit.ry(Parameter("a"), 0)
circuit.rz(Parameter("b"), 0)
circuit.cx(1, 2)
circuit.cx(0, 1)
circuit.h(0)
circuit.measure([0, 1], alpha)
circuit.measure([2], beta)

number_of_params = 100

# Define a sweep over parameter values, where the second axis is over.
# the two parameters in the circuit.
params = np.vstack([
    np.linspace(-np.pi, np.pi, number_of_params),
    np.linspace(-4 * np.pi, 4 * np.pi, number_of_params)
]).T
 
# Instantiate a new statevector simulation based sampler object.
#sampler = QrStatevectorSampler(backend = qr_backend)
sampler = QrSamplerV2(backend = qr_backend)
 
# Start a job that will return shots for all 100 parameter value sets.
pub = (circuit, params)
job = sampler.run([pub], shots=256)
 
# Extract the result for the 0th pub (this example only has one pub).
result = job.result()[0]
 
# There is one BitArray object for each ClassicalRegister in the
# circuit. Here, we can see that the BitArray for alpha contains data
# for all 100 sweep points, and that it is indeed storing data for 2
# bits over 256 shots.


assert result.data.alpha.shape == (number_of_params,)
assert result.data.alpha.num_bits == 2
assert result.data.alpha.num_shots == 256
 
# We can work directly with a binary array in performant applications.
raw = result.data.alpha.array
 
# For small registers where it is anticipated to have many counts
# associated with the same bitstrings, we can turn the data from,
# for example, the 22nd sweep index into a dictionary of counts.
counts = result.data.alpha.get_counts(22)
print(counts)

# Or, convert into a list of bitstrings that preserve shot order.
bitstrings = result.data.alpha.get_bitstrings(22)
print(bitstrings)

assert result.data.beta.num_bits == 1
assert result.data.beta.num_shots == 256
 
# We can work directly with a binary array in performant applications.
raw = result.data.beta.array
 
# For small registers where it is anticipated to have many counts
# associated with the same bitstrings, we can turn the data from,
# for example, the 22nd sweep index into a dictionary of counts.
counts = result.data.beta.get_counts(23)
print(counts)

# Or, convert into a list of bitstrings that preserve shot order.
bitstrings = result.data.beta.get_bitstrings(99)
print(bitstrings)

{'10': 152, '00': 104}
['10', '00', '10', '10', '10', '10', '10', '10', '00', '10', '10', '00', '00', '00', '10', '00', '00', '10', '10', '10', '10', '10', '00', '10', '10', '00', '10', '00', '00', '00', '00', '00', '10', '00', '10', '00', '10', '10', '10', '10', '10', '10', '00', '00', '10', '10', '10', '00', '00', '00', '00', '00', '00', '10', '00', '10', '00', '00', '10', '10', '00', '10', '00', '10', '00', '00', '10', '00', '00', '00', '10', '10', '00', '10', '00', '00', '10', '00', '00', '00', '10', '00', '10', '00', '10', '10', '00', '00', '10', '10', '00', '00', '00', '00', '00', '10', '00', '10', '10', '00', '10', '00', '00', '10', '10', '10', '10', '10', '10', '10', '10', '10', '00', '10', '10', '00', '10', '00', '00', '10', '10', '10', '10', '00', '10', '10', '10', '10', '10', '10', '00', '00', '10', '10', '10', '00', '00', '10', '10', '00', '10', '10', '10', '10', '10', '10', '00', '00', '10', '00', '10', '10', '10', '10', '10', '10', '00', '10', '10', '00', '00', '10', '00'